In [82]:
import pymupdf
import fitz
import re

import firebase_admin
from firebase_admin import credentials
from firebase_admin import firestore
import os 
import datetime 

In [83]:
PDF_FILE_PATH = "./food_regulations.pdf"
SERVICE_ACCOUNT_KEY_PATH = "./arcane-legacy-457717-u2-c427e51e0a61.json"
APP_ID = "food-label-app"

In [84]:
def initialize_firestore(service_account_path):
    """Initializes the Firebase Admin SDK and Firestore."""
    try:
        if not os.path.exists(service_account_path):
            print(f"ERROR: Service account key file not found at '{service_account_path}'.")
            print("Please download it from your Firebase project settings and update SERVICE_ACCOUNT_KEY_PATH.")
            return None
        if not firebase_admin._apps: # Check if already initialized
            cred = credentials.Certificate(service_account_path)
            firebase_admin.initialize_app(cred)
        db = firestore.client()
        print("Firebase Admin SDK initialized and Firestore client obtained.")
        return db
    except Exception as e:
        print(f"Error initializing Firebase: {e}")
        print("Please ensure your service account key path is correct and the file is valid.")
        return None

In [85]:
def extract_text_from_pdf(pdf_path):
    """Extracts text from all pages of a PDF file."""
    if not os.path.exists(pdf_path):
        print(f"ERROR: PDF file not found at '{pdf_path}'.")
        print("Please ensure the PDF_FILE_PATH is correct.")
        return []
        
    doc_text_pages = []
    try:
        doc = fitz.open(pdf_path)
        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            # Using "blocks" can sometimes give better structured text, 
            # especially if trying to avoid headers/footers in a more complex way.
            # For now, sticking to "text" for simplicity, but this is an area for refinement.
            # text = page.get_text("text") 
            
            # Alternative: try to get text blocks to have more control over layout
            blocks = page.get_text("blocks")
            page_text_elements = []
            for b in blocks: # b is a tuple (x0, y0, x1, y1, "lines in block", block_no, block_type)
                # block_type 0 is text, 1 is image.
                # We are interested in text blocks.
                if b[6] == 0: 
                    page_text_elements.append(b[4].strip()) # b[4] contains the text lines
            
            text = "\n".join(page_text_elements)
            doc_text_pages.append(text)

        doc.close()
        print(f"Successfully extracted text from {len(doc_text_pages)} pages in '{os.path.basename(pdf_path)}'.")
    except Exception as e:
        print(f"Error extracting text from PDF '{os.path.basename(pdf_path)}': {e}")
        return []
    return doc_text_pages

In [86]:
def clean_text_chunk(text_chunk):
    """Basic cleaning for a text chunk."""
    if not text_chunk:
        return ""
    # Normalize newlines: replace multiple newlines (and surrounding whitespace) with a single newline
    cleaned_text = re.sub(r'\s*\n\s*(\n\s*)+', '\n', text_chunk)
    # Normalize spaces: replace multiple spaces with a single space, but preserve newlines for structure
    lines = cleaned_text.split('\n')
    processed_lines = []
    for line in lines:
        line = re.sub(r'[ \t]+', ' ', line).strip() # Replace multiple spaces/tabs with one and strip
        if line: # Keep line if it's not empty after stripping
            # Simple heuristic to try and filter out common page footers like "Updated until ..." or just page numbers
            if not re.match(r"^(Updated until December \d{4})$", line, re.IGNORECASE) and \
               not re.match(r"^\d+$", line.strip()): # Avoid lines that are only page numbers
                processed_lines.append(line)
    return "\n".join(processed_lines)

In [87]:
def chunk_text_by_regulation(pages_text, pdf_filename):
    """
    Chunks the document text based on "Regulation X." patterns.
    Each chunk starts with the "Regulation X." heading.
    """
    full_text = "\n".join(pages_text)
    chunks = []
    
    # Regex to find "Regulation X." or "Regulation XA." patterns (case-insensitive, multiline, start of line)
    pattern = re.compile(r"(Regulation\s+\d+[A-Z]?\.?\s+.+?)(?=\nRegulation\s+\d+[A-Z]?\.|\Z)", re.IGNORECASE | re.MULTILINE)

    
    matches = list(pattern.finditer(full_text))
    
    source_doc_firestore_id = re.sub(r'[^\w-]', '_', os.path.splitext(pdf_filename)[0])

    if not matches:
        print("No 'Regulation X.' patterns found. Storing the whole document as one chunk after cleaning.")
        # Apply cleaning to the full text before storing it as a single chunk
        cleaned_full_text = clean_text_chunk(full_text) 
        if cleaned_full_text: 
            chunks.append({
                "id_title": "Full_Document_Content_0", 
                "display_title": "Full Document Content",
                "content": cleaned_full_text,
                "original_filename": pdf_filename,
                "source_document_firestore_id": source_doc_firestore_id 
            })
        return chunks

    print(f"\nFound {len(matches)} potential regulation headings. Starting chunking process...")
    for i, match in enumerate(matches):
        # display_title_raw is the "Regulation X." part (e.g., "Regulation 1.")
        display_title_raw = match.group(1).strip() 
        
        # id_title is the sanitized version for Firestore document ID
        id_title = re.sub(r'[^\w\s-]', '', display_title_raw) 
        id_title = re.sub(r'\s+', '_', id_title).replace('.', '') 

        # start_index_for_full_chunk is the beginning of the current "Regulation X." line in full_text
        start_index_for_full_chunk = match.start()
        # end_index_for_full_chunk is the beginning of the *next* "Regulation Y." line, or end of document
        end_index_for_full_chunk = matches[i+1].start() if i + 1 < len(matches) else len(full_text)
            
        # This slice should contain the current regulation's title AND all its body text
        chunk_content_raw_with_title = full_text[start_index_for_full_chunk : end_index_for_full_chunk]
        
        # --- DEBUGGING PRINTS ---
        print(f"\n----------------------------------------------------")
        print(f"Processing for: {display_title_raw} (Sanitized ID: {id_title})")
        print(f"Matched line (match.group(0)): {match.group(0).strip()}") # The full line that the regex matched
        print(f"Slice in full_text: from index {start_index_for_full_chunk} to {end_index_for_full_chunk}")
        print(f"Length of raw slice: {len(chunk_content_raw_with_title)}")
        
        print(f"\nRAW CONTENT PREVIEW (first 500 chars of chunk_content_raw_with_title before initial strip):")
        print(chunk_content_raw_with_title[:500].strip()) # .strip() here just for cleaner debug printing
        
        # Clean the extracted slice
        cleaned_content_for_storage = clean_text_chunk(chunk_content_raw_with_title)
        
        print(f"\nCLEANED CONTENT PREVIEW (first 500 chars for {display_title_raw}):")
        print(cleaned_content_for_storage[:500])
        print(f"Length of cleaned content: {len(cleaned_content_for_storage)}")
        print(f"----------------------------------------------------")
        # --- END DEBUGGING PRINTS ---
        
        if cleaned_content_for_storage: 
            # Ensure the content is not *just* the title itself if possible,
            # though the title is part of the content by design here.
            # A more sophisticated check could compare cleaned_content_for_storage with just the cleaned title.
            if len(cleaned_content_for_storage.strip()) > len(match.group(0).strip()) + 10 or \
               '\n' in cleaned_content_for_storage.strip() or \
               len(matches) == 1: # If only one chunk, it's the whole doc
                chunks.append({
                    "id_title": id_title, 
                    "display_title": display_title_raw, 
                    "content": cleaned_content_for_storage, 
                    "original_filename": pdf_filename,
                    "source_document_firestore_id": source_doc_firestore_id
                })
            else:
                print(f"WARNING: Chunk for {display_title_raw} seems to contain only the title or very little extra content. Skipping storage for this chunk or review logic.")

        
    print(f"\nSuccessfully processed {len(chunks)} chunks to be stored (after filtering).")
    return chunks

In [88]:
def chunk_text_by_page(pages_text):
    """
    Chunks the document text page by page.

    Args:
        pages_text (list): A list of strings, where each string is the text of a page.

    Returns:
        list: A list of dictionaries, where each dictionary represents a page chunk.
    """
    chunks = []
    for i, page_content in enumerate(pages_text):
        cleaned_content = clean_text_chunk(page_content)
        if cleaned_content: # Only add if there's actual content after cleaning
            chunks.append({
                "title": f"Page_{i+1}",
                "content": cleaned_content
            })
    print(f"Successfully created {len(chunks)} chunks (one per page).")
    return chunks

In [89]:
def store_chunks_in_firestore(db, chunks, pdf_filename_original):
    """Stores the processed chunks into Firestore under a parent document representing the PDF."""
    if not db:
        print("Firestore client not available. Skipping storage.")
        return False
    if not chunks:
        print("No chunks to store.")
        return False

    # Sanitize PDF filename to be a valid Firestore document ID
    pdf_doc_id_sanitized = "food-label-app"

    regulations_collection_name = "processed_regulations"
    pdf_parent_doc_ref = db.collection(f"artifacts/{APP_ID}/public/data/{regulations_collection_name}").document(pdf_doc_id_sanitized)

    print(f"\nStoring chunks for '{pdf_filename_original}' under Firestore document ID: '{pdf_doc_id_sanitized}' (within APP_ID: '{APP_ID}')")
    
    pdf_metadata = {
        "original_filename": pdf_filename_original,
        "processed_at": firestore.SERVER_TIMESTAMP, 
        "number_of_chunks": len(chunks),
        "app_id_source": APP_ID 
    }
    
    try:
        pdf_parent_doc_ref.set(pdf_metadata)
        print(f"Metadata for '{pdf_filename_original}' stored successfully in app '{APP_ID}'.")

        chunks_subcollection_ref = pdf_parent_doc_ref.collection("regulation_details")
        
        batch = db.batch()
        stored_count = 0
        operations_in_current_batch = 0 # Counter for operations in the current batch

        for chunk_data in chunks:
            chunk_doc_id = chunk_data.get("id_title", f"chunk_{stored_count}") 
            if not chunk_doc_id: 
                chunk_doc_id = f"untitled_chunk_{stored_count}"

            chunk_doc_ref = chunks_subcollection_ref.document(chunk_doc_id)
            
            data_to_store = {
                "display_title": chunk_data.get("display_title"),
                "content": chunk_data.get("content"),
                "original_filename_ref": chunk_data.get("original_filename"), 
                "source_document_ref_id": chunk_data.get("source_document_firestore_id")
            }
            batch.set(chunk_doc_ref, data_to_store)
            stored_count += 1
            operations_in_current_batch += 1
            
            # Firestore batch limit is 500 operations. Commit slightly before to be safe.
            if operations_in_current_batch >= 499: 
                print(f"Committing batch of {operations_in_current_batch} chunks...")
                batch.commit()
                batch = db.batch() # Start a new batch
                operations_in_current_batch = 0 # Reset counter for the new batch
        
        # Commit any remaining chunks in the last batch
        if operations_in_current_batch > 0:
            print(f"Committing final batch of {operations_in_current_batch} chunks...")
            batch.commit()

        print(f"Successfully stored {stored_count} chunks for '{pdf_filename_original}' in Firestore under app '{APP_ID}'.")
        return True
    except Exception as e:
        print(f"Error storing chunks in Firestore: {e}")
        return False



In [90]:
print("Starting PDF processing script...")
print(f"Using APP_ID: {APP_ID}") # Confirm the APP_ID being used


db_client = initialize_firestore(SERVICE_ACCOUNT_KEY_PATH)

if db_client:
    pdf_original_filename = os.path.basename(PDF_FILE_PATH)
    extracted_pages = extract_text_from_pdf(PDF_FILE_PATH)

    if extracted_pages:
        processed_chunks = chunk_text_by_regulation(extracted_pages, pdf_original_filename)

        if processed_chunks:
            success = store_chunks_in_firestore(db_client, processed_chunks, pdf_original_filename)
            if success:
                print("\nPDF processing and Firestore upload completed successfully.")
            else:
                print("\nPDF processing completed, but there were errors during Firestore upload.")
        else:
            print("No chunks were generated from the PDF content for storage.")
    else:
        print("Text extraction failed. Cannot proceed with chunking and storage.")
else:
    print("Firestore initialization failed. Cannot proceed.")

print("Script finished.")

Starting PDF processing script...
Using APP_ID: food-label-app
Firebase Admin SDK initialized and Firestore client obtained.
Successfully extracted text from 168 pages in 'food_regulations.pdf'.

Found 404 potential regulation headings. Starting chunking process...

----------------------------------------------------
Processing for: Regulation 1. Citation, commencement and application. (Sanitized ID: Regulation_1_Citation_commencement_and_application)
Matched line (match.group(0)): Regulation 1. Citation, commencement and application.
Slice in full_text: from index 171 to 558
Length of raw slice: 387

RAW CONTENT PREVIEW (first 500 chars of chunk_content_raw_with_title before initial strip):
Regulation 1. Citation, commencement and application. 
Regulation 2. Interpretation.
PART II  
WARRANTY 
 
Regulation 3. Food which requires a written warranty from manufacturer, etc.

PART IIA  
APPROVAL FOR SALE OF FOOD OBTAINED THROUGH MODERN BIOTECHNOLOGY 
 
Regulation 3A. Approval for sale of